In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "tiendung"
CODAPATH = Path("/kaggle/working/codapath")
if (CODAPATH / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(CODAPATH), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "pull", "--ff-only", "origin", REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f"{CODAPATH} exists but is not a Git repository")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(CODAPATH)])
actual_branch = subprocess.check_output(["git", "-C", str(CODAPATH), "branch", "--show-current"], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print("repo:", CODAPATH, "| branch:", actual_branch)

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public; Kaggle Internet must be enabled.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import numpy as np
import torch

from set_up import set_seed
from load_data import get_data_loaders
from model import DINOv2Extractor, extract_image_features
from trainer import load_model
from evaluate import evaluate_model

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])
PATHMNIST_PATH  = str(DATA_ROOT / "pathmnist_224.npz")
HISTOSET_PATH   = str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14")
SKINTISSUE_PATH = str(DATA_ROOT / "SkinTissue/SkinTissue/tiles")

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
# ---- EDIT THIS CELL ----
CONFIG_PATH = "config/config.yaml"
DATASET = "pathmnist"
RUN_NAME = "nucleus_cellvit_embedding_disagreement"
SEED = 42
# Attach the output dataset produced by run_al.ipynb.
CHECKPOINT_ROOT = "/kaggle/input/EDIT_RUN_OUTPUT_SLUG/checkpoints"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

random_seed = SEED
device = torch.device(config["device"])
data_path = Path(DATA_DICT[DATASET])
checkpoint_dir = Path(CHECKPOINT_ROOT) / DATASET
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert checkpoint_dir.is_dir(), (
    f"Missing checkpoint directory: {checkpoint_dir}. Attach run_al output and edit CHECKPOINT_ROOT."
)
assert torch.cuda.is_available(), "Attach a Kaggle GPU before evaluation"

In [ ]:
set_seed(random_seed)

_, test_loader, class_names = get_data_loaders(DATA_DICT[DATASET], random_seed, verbose=True)
test_dataset = test_loader.dataset
test_labels = (
    test_dataset.lbl
    if hasattr(test_dataset, "lbl")
    else np.array(test_dataset.dataset.targets)[test_dataset.indices]
)

vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
extractor = DINOv2Extractor(model_name=vit_name).to(device)
test_features = extract_image_features(test_loader, extractor, device)
del extractor

In [ ]:
for budget in config["cumulative_budget"]:
    checkpoint_file = checkpoint_dir / f"{RUN_NAME}_probe_budget_{budget}.pt"
    assert checkpoint_file.is_file(), f"Missing checkpoint: {checkpoint_file}"
    probe = load_model(str(checkpoint_file), device)

    print(f"=== {DATASET} | {RUN_NAME} | budget={budget} ===")
    evaluate_model(probe, test_features, test_labels, device)
    print()